In [1]:
import sys
sys.path.append("../../")
from src.data_handle.embed import get_dataloaders, get_embed, DatasetConfig, DataLoaderConfig

In [2]:
dataset_cfg = DatasetConfig(
    data_dir="/vol/biodata/data/Mammo/EMBED/pngs/1024x768",
    split_dir="/vol/biomedic3/tx1215/mamo-flow/assets/embed_splits_v1",
    parents=["age", "view", "density", "scanner", "cview"],
    img_height=512,
    img_width=384,
    img_channels=1,
)
loader_cfg = DataLoaderConfig(
    bs=12,
    num_workers=12,
    prefetch_factor=2,
    seed=0,
)

In [9]:
import pandas as pd
from src.data_handle.embed import load_split_csvs

split_dfs = load_split_csvs(dataset_cfg.split_dir)
for split, df in split_dfs.items():
    print(f"\n--- {split} ({len(df)} rows) ---")
    print(f"Columns: {list(df.columns)}")
    if "tissueden" in df.columns:
        print(f"density: {df['tissueden'].value_counts(normalize=True).to_dict()}")
    if "ViewPosition" in df.columns:
        print(f"view:    {df['ViewPosition'].value_counts(normalize=True).to_dict()}")
    if "cview" in df.columns:
        print(f"Cview:    {df['cview'].value_counts(normalize=True).to_dict()}")


--- train (237746 rows) ---
Columns: ['cache_idx', 'empi_anon', 'tissueden', 'study_date_anon', 'acc_anon', 'asses', 'age_at_study', 'image_path', 'FinalImageType', 'ImageLateralityFinal', 'ViewPosition', 'Manufacturer', 'ManufacturerModelName', 'image_path_orig', 'shortpath', 'manufacturer_domain', 'density', 'scanner', 'view', 'cview', 'age']
density: {'B': 0.4208903619829566, 'C': 0.4142782633566916, 'A': 0.11224163603173135, 'D': 0.05258973862862046}
view:    {'MLO': 0.5307008319803488, 'CC': 0.4692991680196512}
Cview:    {0: 1.0}

--- valid (22693 rows) ---
Columns: ['cache_idx', 'empi_anon', 'tissueden', 'study_date_anon', 'acc_anon', 'asses', 'age_at_study', 'image_path', 'FinalImageType', 'ImageLateralityFinal', 'ViewPosition', 'Manufacturer', 'ManufacturerModelName', 'image_path_orig', 'shortpath', 'manufacturer_domain', 'density', 'scanner', 'view', 'cview', 'age']
density: {'B': 0.42101088441369583, 'C': 0.41083153395320143, 'A': 0.11219318732648835, 'D': 0.0559643943066143

In [4]:
print(split_dfs['train'].columns.tolist())

print(split_dfs['train']['cache_idx'])

['cache_idx', 'empi_anon', 'tissueden', 'study_date_anon', 'acc_anon', 'asses', 'age_at_study', 'image_path', 'FinalImageType', 'ImageLateralityFinal', 'ViewPosition', 'Manufacturer', 'ManufacturerModelName', 'image_path_orig', 'shortpath', 'manufacturer_domain', 'density', 'scanner', 'view', 'cview', 'age']
0              0
1              1
2              2
3              3
4              4
           ...  
237741    237741
237742    237742
237743    237743
237744    237744
237745    237745
Name: cache_idx, Length: 237746, dtype: int64


In [5]:
datasets = get_embed(dataset_cfg)
print({k: len(v) for k, v in datasets.items()})

dataloaders = get_dataloaders(loader_cfg, datasets)
print(dataloaders)

batch = next(iter(dataloaders["train"]))
print(batch.keys())
print("x shape:", batch["x"].shape)
print("x range:", batch["x"].min(), batch["x"].max())
# print("shortpath:", batch["shortpath"][:2])
print("pa:", batch["pa"].keys())

{'train': 237746, 'valid': 22693, 'test': 36934}
{'train': <torch.utils.data.dataloader.DataLoader object at 0x7ee890b378c0>, 'valid': <torch.utils.data.dataloader.DataLoader object at 0x7ee890b579d0>, 'test': <torch.utils.data.dataloader.DataLoader object at 0x7ee8807b4050>}
dict_keys(['x', 'pa', 'shortpath'])
x shape: torch.Size([12, 1, 512, 384])
x range: tensor(0.) tensor(1.)
pa: dict_keys(['age', 'view', 'density', 'scanner', 'cview'])


In [ ]:
for k in batch["pa"].keys():
    print(f"pa[{k}] shape: {batch['pa'][k].shape}, range: {batch['pa'][k].min()} - {batch['pa'][k].max()}")
print(f"batch['x'] range after normalization: {batch['x'].min()} - {batch['x'].max()}")

pa[age] shape: torch.Size([12, 1]), range: 0.4432671368122101 - 0.8234803080558777
pa[view] shape: torch.Size([12, 2]), range: 0.0 - 1.0
pa[density] shape: torch.Size([12, 4]), range: 0.0 - 1.0
pa[scanner] shape: torch.Size([12, 5]), range: 0.0 - 1.0
pa[cview] shape: torch.Size([12, 2]), range: 0.0 - 1.0
batch['x'] range after normalization: 0.0 - 1.0
